# Experiment 15 — Clip Augmentation for Rare Species MLP Probes

**Building on:** exp11 best (cmAP=0.9166) / exp13 (cmAP=0.9170)

**Key idea:** For species with 1-20 soundscape labeled positives, augment MLP probe training
with Perch embeddings from training audio clips (Xeno-canto/iNat, rating>=3.0). This gives
10-50x more positive examples for rare species, helping probe quality.

**Changes vs exp11:**
- Extract Perch embeddings from training clips (up to 50 per rare species)
- Add clip embeddings as additional positives in MLP probe training
- Same LightProtoSSM (focal + OneCycleLR from exp13)
- OOF grid search for blend weights

**Hypothesis:** 27 rare species have 1-20 soundscape positives. More positive examples should improve their per-class AP, lifting overall cmAP.

In [1]:
import os, re, gc, time, warnings, concurrent.futures
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torch.nn as nn
import torch.nn.functional as F
import onnxruntime as ort
from pathlib import Path
from tqdm.auto import tqdm
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import GroupKFold
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier

torch.manual_seed(42)
np.random.seed(42)

PROJECT_ROOT = Path('/Users/jjannik/Development/kaggle/birdclef')
BASE      = PROJECT_ROOT / 'data'
MODEL_DIR = BASE / 'models' / 'perch_tf'
ONNX_PATH = BASE / 'models' / 'perch' / 'perch_v2_no_dft.onnx'
CACHE_DIR = BASE / 'cache'
CACHE_META = CACHE_DIR / 'exp11_perch_meta.csv'
CACHE_NPZ  = CACHE_DIR / 'exp11_perch_arrays.npz'
CLIP_CACHE = CACHE_DIR / 'exp15_clip_embeddings.npz'

SR             = 32_000
WINDOW_SAMPLES = SR * 5
FILE_SAMPLES   = 60 * SR
N_WINDOWS      = 12
N_OOF_SPLITS   = 3

# Species with <= this many soundscape positives get clip augmentation
RARE_THRESHOLD = 20
# Max clips per species
MAX_CLIPS_PER_SPECIES = 50

CFG = dict(
    d_model=128, d_state=16, n_ssm_layers=2, dropout=0.15,
    n_sites=20, meta_dim=16,
    n_epochs=30, lr=5e-4, weight_decay=1e-3, patience=8,
    focal_gamma=1.5, distill_weight=0.15, pos_weight_cap=25.0,
    pca_dim=64, mlp_hidden=(128, 64), min_pos=5, alpha_blend=0.25,
    lambda_prior=0.10,
    # Clip augmentation weight: how many times to repeat clip examples
    clip_repeat_factor=1,
)

_WALL_START = time.time()
print('Setup done')

Setup done


In [2]:
# Data loading
taxonomy          = pd.read_csv(BASE / 'taxonomy.csv')
train_meta        = pd.read_csv(BASE / 'train.csv')
sample_sub        = pd.read_csv(BASE / 'sample_submission.csv')
soundscape_labels = pd.read_csv(BASE / 'train_soundscapes_labels.csv')

PRIMARY_LABELS = sample_sub.columns[1:].tolist()
N_CLASSES      = len(PRIMARY_LABELS)
label_to_idx   = {c: i for i, c in enumerate(PRIMARY_LABELS)}
CLASS_NAME_MAP = taxonomy.set_index('primary_label')['class_name'].to_dict()

FNAME_RE = re.compile(r'BC2026_(?:Train|Test)_(\d+)_(S\d+)_(\d{8})_(\d{6})\.ogg')

def parse_fname(name):
    m = FNAME_RE.match(name)
    if not m: return {'site': 'unknown', 'hour_utc': -1}
    _, site, _, hms = m.groups()
    return {'site': site, 'hour_utc': int(hms[:2])}

def union_labels(series):
    out = set()
    for x in series:
        if pd.notna(x):
            for t in str(x).split(';'):
                t = t.strip()
                if t: out.add(t)
    return sorted(out)

sc = (
    soundscape_labels
    .groupby(['filename', 'start', 'end'])['primary_label']
    .apply(union_labels)
    .reset_index(name='label_list')
)
sc['end_sec'] = pd.to_timedelta(sc['end']).dt.total_seconds().astype(int)
sc['row_id']  = sc['filename'].str.replace('.ogg', '', regex=False) + '_' + sc['end_sec'].astype(str)
_meta = sc['filename'].apply(parse_fname).apply(pd.Series)
sc = pd.concat([sc, _meta], axis=1)

Y_SC = np.zeros((len(sc), N_CLASSES), dtype=np.uint8)
for i, lbls in enumerate(sc['label_list']):
    for lbl in lbls:
        if lbl in label_to_idx: Y_SC[i, label_to_idx[lbl]] = 1

windows_per_file = sc.groupby('filename').size()
LABEL_WINDOWS    = int(windows_per_file.mode().iloc[0])
full_files       = sorted(windows_per_file[windows_per_file == LABEL_WINDOWS].index.tolist())
sc['fully_labeled'] = sc['filename'].isin(full_files)
full_rows = sc[sc['fully_labeled']].sort_values(['filename', 'end_sec']).reset_index(drop=False)

# Identify rare species
soundscape_pos = Y_SC.sum(axis=0)
rare_species   = [PRIMARY_LABELS[i] for i, p in enumerate(soundscape_pos)
                  if 1 <= p <= RARE_THRESHOLD]

print(f'Classes: {N_CLASSES} | Fully-labeled files: {len(full_files)}')
print(f'Rare species (1-{RARE_THRESHOLD} soundscape positives): {len(rare_species)}')

Classes: 234 | Fully-labeled files: 59
Rare species (1-20 soundscape positives): 43


In [3]:
# Load Perch ONNX
import re as _re

_so = ort.SessionOptions(); _so.intra_op_num_threads = 4
ONNX_SESSION    = ort.InferenceSession(str(ONNX_PATH), sess_options=_so, providers=['CPUExecutionProvider'])
ONNX_INPUT_NAME = ONNX_SESSION.get_inputs()[0].name
ONNX_OUT_MAP    = {o.name: i for i, o in enumerate(ONNX_SESSION.get_outputs())}

bc_labels = (pd.read_csv(MODEL_DIR / 'assets' / 'labels.csv')
             .reset_index().rename(columns={'index': 'bc_index', 'inat2024_fsd50k': 'scientific_name'}))
NO_LABEL  = len(bc_labels)
mapping   = taxonomy.merge(bc_labels, on='scientific_name', how='left')
mapping['bc_index'] = mapping['bc_index'].fillna(NO_LABEL).astype(int)
lbl2bc    = mapping.set_index('primary_label')['bc_index']
BC_INDICES    = np.array([int(lbl2bc.loc[c]) for c in PRIMARY_LABELS], dtype=np.int32)
MAPPED_MASK   = BC_INDICES != NO_LABEL
MAPPED_POS    = np.where(MAPPED_MASK)[0].astype(np.int32)
MAPPED_BC_IDX = BC_INDICES[MAPPED_MASK].astype(np.int32)

# Proxy map for unmapped species
proxy_map = {}
for _, row in taxonomy[taxonomy['primary_label'].isin(
        [PRIMARY_LABELS[i] for i in np.where(~MAPPED_MASK)[0]])].iterrows():
    genus = str(row['scientific_name']).split()[0]
    hits  = bc_labels[bc_labels['scientific_name'].astype(str).str.match(rf'^{_re.escape(genus)}\s', na=False)]
    if len(hits) > 0:
        proxy_map[label_to_idx[row['primary_label']]] = hits['bc_index'].astype(int).tolist()
proxy_map = {k: v for k, v in proxy_map.items()
             if CLASS_NAME_MAP.get(PRIMARY_LABELS[k]) in {'Amphibia', 'Insecta', 'Aves'}}

print(f'ONNX loaded. Mapped: {MAPPED_MASK.sum()}/{N_CLASSES}')

ONNX loaded. Mapped: 203/234

In [4]:
# Extract Perch embeddings from training clips

def read_clip(path, n_samples=WINDOW_SAMPLES):
    try:
        y, sr = sf.read(path, dtype='float32', always_2d=False)
        if y.ndim == 2: y = y.mean(axis=1)
        if len(y) > n_samples: y = y[:n_samples]
        else: y = np.pad(y, (0, n_samples - len(y)))
        return y
    except Exception:
        return np.zeros(n_samples, dtype=np.float32)

def extract_clip_embeddings(paths, batch_size=64, verbose=True):
    n = len(paths)
    embs   = np.zeros((n, 1536), dtype=np.float32)
    scores = np.zeros((n, N_CLASSES), dtype=np.float32)
    itr = range(0, n, batch_size)
    if verbose: itr = tqdm(itr, desc='Clip Perch')
    for start in itr:
        batch = paths[start:start + batch_size]
        x = np.stack([read_clip(p) for p in batch])  # (B, 160000)
        outs   = ONNX_SESSION.run(None, {ONNX_INPUT_NAME: x})
        logits = outs[ONNX_OUT_MAP['label']].astype(np.float32)
        emb    = outs[ONNX_OUT_MAP['embedding']].astype(np.float32)
        b = len(batch)
        scores[start:start+b, MAPPED_POS] = logits[:, MAPPED_BC_IDX]
        for pos_idx, bc_idxs in proxy_map.items():
            scores[start:start+b, pos_idx] = logits[:, np.array(bc_idxs)].max(axis=1)
        embs[start:start+b] = emb
        del x, outs, logits, emb; gc.collect()
    return embs, scores


# Build clip set
good_clips = train_meta[(train_meta['rating'] >= 3.0) &
                        (train_meta['primary_label'].isin(rare_species))].copy()
# Cap at MAX_CLIPS_PER_SPECIES per species, prioritize high ratings
good_clips = (good_clips.sort_values('rating', ascending=False)
              .groupby('primary_label').head(MAX_CLIPS_PER_SPECIES)
              .reset_index(drop=True))

print(f'Clips to process: {len(good_clips)} from {good_clips["primary_label"].nunique()} species')
clip_paths  = [BASE / 'train_audio' / fn for fn in good_clips['filename']]
clip_labels = np.array([label_to_idx[lbl] for lbl in good_clips['primary_label']])

# Check which files actually exist
exists_mask = np.array([p.exists() for p in clip_paths])
print(f'Files found: {exists_mask.sum()} / {len(clip_paths)}')
clip_paths  = [p for p, e in zip(clip_paths, exists_mask) if e]
clip_labels = clip_labels[exists_mask]
good_clips  = good_clips[exists_mask].reset_index(drop=True)

Clips to process: 1032 from 27 species
Files found: 1032 / 1032


In [5]:
# Build or load clip embedding cache
if CLIP_CACHE.exists():
    print('Loading clip cache...')
    _c = np.load(CLIP_CACHE)
    clip_emb   = _c['embs'].astype(np.float32)
    clip_sc    = _c['scores'].astype(np.float32)
    clip_labels_cached = _c['labels'].astype(np.int32)
    clip_labels = clip_labels_cached
else:
    print(f'Extracting Perch for {len(clip_paths)} clips...')
    t0 = time.time()
    clip_emb, clip_sc = extract_clip_embeddings(clip_paths, batch_size=64, verbose=True)
    print(f'Done in {time.time()-t0:.1f}s')
    np.savez(CLIP_CACHE, embs=clip_emb, scores=clip_sc, labels=clip_labels)
    print('Cache saved.')

print(f'clip_emb: {clip_emb.shape}  clip_sc: {clip_sc.shape}')

Extracting Perch for 1032 clips...


Clip Perch:   0%|          | 0/17 [00:00<?, ?it/s]

Done in 40.9s
Cache saved.
clip_emb: (1032, 1536)  clip_sc: (1032, 234)


In [6]:
# Load soundscape Perch cache
def _pick_array(arr, keys, ncols):
    for k in keys:
        if k in arr.files: return arr[k]
    for k in arr.files:
        v = arr[k]
        if v.ndim == 2 and v.shape[1] == ncols: return v
    raise KeyError(f'{keys} not in {arr.files}')

meta_tr = pd.read_csv(CACHE_META)
_arr    = np.load(CACHE_NPZ)
sc_tr   = _pick_array(_arr, ['scores', 'sc', 'logits', 'arr_0'], N_CLASSES).astype(np.float32)
emb_tr  = _pick_array(_arr, ['embs', 'emb', 'embeddings', 'arr_1'], 1536).astype(np.float32)
row_id_to_index = full_rows.set_index('row_id')['index']
Y_FULL_aligned  = Y_SC[row_id_to_index.loc[meta_tr['row_id']].to_numpy()]
print(f'Soundscape cache: sc_tr={sc_tr.shape}  emb_tr={emb_tr.shape}')

Soundscape cache: sc_tr=(708, 234)  emb_tr=(708, 1536)


In [7]:
# Metrics + loss
def macro_auc(y_true, y_score):
    keep = y_true.sum(axis=0) > 0
    return roc_auc_score(y_true[:, keep], y_score[:, keep], average='macro')

def padded_cmap(y_true, y_pred, pad=5):
    aps = []
    for c in range(y_true.shape[1]):
        yt = np.concatenate([y_true[:, c], np.ones(pad)])
        yp = np.concatenate([y_pred[:, c], np.ones(pad)])
        aps.append(average_precision_score(yt, yp))
    return float(np.mean(aps))

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -30, 30)))

def focal_bce(logits, targets, gamma=1.5, pos_weight=None):
    bce = F.binary_cross_entropy_with_logits(logits, targets, pos_weight=pos_weight, reduction='none')
    p_t = torch.exp(-bce)
    return ((1 - p_t) ** gamma * bce).mean()

print('Metrics ready')

Metrics ready


In [8]:
# Post-processing helpers — same as exp11

def build_prior_tables(sc_df, Y_labels):
    sc_df = sc_df.reset_index(drop=True)
    global_p = Y_labels.mean(axis=0).astype(np.float32)
    site_keys = sorted(sc_df['site'].dropna().astype(str).unique())
    site_to_i = {k: i for i, k in enumerate(site_keys)}
    site_p = np.zeros((len(site_keys), Y_labels.shape[1]), dtype=np.float32)
    site_n = np.zeros(len(site_keys), dtype=np.float32)
    for s in site_keys:
        i = site_to_i[s]; mask = sc_df['site'].astype(str).values == s
        site_n[i] = mask.sum(); site_p[i] = Y_labels[mask].mean(axis=0)
    hour_keys = sorted(sc_df['hour_utc'].dropna().astype(int).unique())
    hour_to_i = {h: i for i, h in enumerate(hour_keys)}
    hour_p = np.zeros((len(hour_keys), Y_labels.shape[1]), dtype=np.float32)
    hour_n = np.zeros(len(hour_keys), dtype=np.float32)
    for h in hour_keys:
        i = hour_to_i[h]; mask = sc_df['hour_utc'].astype(int).values == h
        hour_n[i] = mask.sum(); hour_p[i] = Y_labels[mask].mean(axis=0)
    return {'global_p': global_p,
            'site_to_i': site_to_i, 'site_p': site_p, 'site_n': site_n,
            'hour_to_i': hour_to_i, 'hour_p': hour_p, 'hour_n': hour_n}

def apply_prior(scores, sites, hours, tables, lambda_prior=0.10):
    eps = 1e-4; out = scores.copy()
    p   = np.tile(tables['global_p'], (len(scores), 1))
    for i, h in enumerate(hours):
        h = int(h)
        if h in tables['hour_to_i']:
            j = tables['hour_to_i'][h]; nh = tables['hour_n'][j]
            w = nh / (nh + 8.0)
            p[i] = w * tables['hour_p'][j] + (1 - w) * tables['global_p']
    for i, s in enumerate(sites):
        s = str(s)
        if s in tables['site_to_i']:
            j = tables['site_to_i'][s]; ns = tables['site_n'][j]
            w = ns / (ns + 8.0)
            p[i] = w * tables['site_p'][j] + (1 - w) * p[i]
    p = np.clip(p, eps, 1 - eps)
    out += lambda_prior * (np.log(p) - np.log1p(-p))
    return out.astype(np.float32)

def adaptive_delta_smooth(probs, n_windows=N_WINDOWS, base_alpha=0.10):
    result = probs.copy(); view = probs.reshape(-1, n_windows, probs.shape[1])
    out    = result.reshape(-1, n_windows, probs.shape[1])
    for t in range(n_windows):
        conf = view[:, t, :].max(axis=-1, keepdims=True)
        alpha = base_alpha * (1.0 - conf)
        nbr = (view[:, t-1, :] + view[:, t+1, :]) / 2.0 if 0 < t < n_windows-1 else \
              (view[:, t, :] + view[:, t+1, :]) / 2.0 if t == 0 else \
              (view[:, t-1, :] + view[:, t, :]) / 2.0
        out[:, t, :] = (1.0 - alpha) * view[:, t, :] + alpha * nbr
    return result

def file_confidence_scale(probs, n_windows=N_WINDOWS, top_k=2, power=0.05):
    view = probs.reshape(-1, n_windows, probs.shape[1])
    top_mean = np.sort(view, axis=1)[:, -top_k:, :].mean(axis=1, keepdims=True)
    return (view * np.power(top_mean, power)).reshape(probs.shape)

TEXTURE_TAXA = {'Amphibia', 'Insecta'}
temperatures = np.ones(N_CLASSES, dtype=np.float32)
for ci, lbl in enumerate(PRIMARY_LABELS):
    temperatures[ci] = 0.95 if CLASS_NAME_MAP.get(lbl, 'Aves') in TEXTURE_TAXA else 1.10

print('Helpers ready')

Helpers ready


In [9]:
# MLP probes with clip augmentation

def build_class_freq_weights(Y, cap=10.0):
    pos_count = Y.sum(axis=0).astype(np.float32) + 1.0
    freq      = pos_count / Y.shape[0]
    weights   = np.clip(1.0 / (freq ** 0.5), 1.0, cap)
    return (weights / weights.mean()).astype(np.float32)

def build_seq_features(scores_col, n_windows=N_WINDOWS):
    x     = scores_col.reshape(-1, n_windows)
    prev  = np.concatenate([x[:, :1], x[:, :-1]], axis=1).reshape(-1)
    next_ = np.concatenate([x[:, 1:], x[:, -1:]],  axis=1).reshape(-1)
    mean  = np.repeat(x.mean(axis=1), n_windows)
    max_  = np.repeat(x.max(axis=1),  n_windows)
    std   = np.repeat(x.std(axis=1),  n_windows)
    return prev, next_, mean, max_, std

def build_clip_seq_features(scores_col):
    # For single-clip (no sequence), context = self
    prev  = scores_col.copy()
    next_ = scores_col.copy()
    mean  = scores_col.copy()
    max_  = scores_col.copy()
    std   = np.zeros_like(scores_col)
    return prev, next_, mean, max_, std

def train_mlp_probes_with_clips(emb, scores_raw, Y,
                                 clip_emb_pca, clip_scores, clip_class_idx,
                                 scaler, pca, class_weights,
                                 pca_dim=64, alpha_blend=0.25, min_pos=5,
                                 clip_repeat=1):
    probe_models = {}
    active = np.where(Y.sum(axis=0) >= min_pos)[0]
    MAX_ROWS = 2000

    for ci in active:
        y = Y[:, ci]
        if y.sum() == 0 or y.sum() == len(y): continue

        # Soundscape features
        Z = pca.transform(scaler.transform(emb)).astype(np.float32)
        prev, next_, mean, max_, std = build_seq_features(scores_raw[:, ci])
        X = np.hstack([Z, scores_raw[:, ci:ci+1],
                       prev[:, None], next_[:, None],
                       mean[:, None], max_[:, None], std[:, None]])

        n_pos = int(y.sum()); pos_idx = np.where(y == 1)[0]
        w = float(class_weights[ci])
        repeat = min(max(1, int(round(w * (len(y) - n_pos) / max(n_pos, 1)))), 8)
        if n_pos * repeat + len(y) > MAX_ROWS:
            repeat = max(1, (MAX_ROWS - len(y)) // max(n_pos, 1))

        X_bal = np.vstack([X, np.tile(X[pos_idx], (repeat, 1))])
        y_bal = np.concatenate([y, np.ones(n_pos * repeat, dtype=y.dtype)])

        # Add clip augmentation for rare species
        clip_ci_mask = clip_class_idx == ci
        if clip_ci_mask.sum() > 0 and y.sum() <= RARE_THRESHOLD:
            clip_X_ci   = clip_emb_pca[clip_ci_mask]
            clip_sc_ci  = clip_scores[clip_ci_mask, ci]
            cp, cn, cm, cmx, cstd = build_clip_seq_features(clip_sc_ci)
            X_clip = np.hstack([clip_X_ci, clip_sc_ci[:, None],
                                cp[:, None], cn[:, None], cm[:, None], cmx[:, None], cstd[:, None]])
            # Add with clip_repeat copies (default 1 = same weight as soundscape positives)
            for _ in range(max(1, clip_repeat)):
                X_bal = np.vstack([X_bal, X_clip])
                y_bal = np.concatenate([y_bal, np.ones(len(X_clip), dtype=y.dtype)])

        clf = MLPClassifier(
            hidden_layer_sizes=CFG['mlp_hidden'], activation='relu',
            max_iter=200, early_stopping=True,
            validation_fraction=0.15, n_iter_no_change=15,
            random_state=42, learning_rate_init=5e-4, alpha=0.005,
        )
        clf.fit(X_bal, y_bal)
        probe_models[ci] = clf

    return probe_models

def apply_mlp_probes(emb_test, scores_test, probe_models, scaler, pca, alpha_blend=0.25):
    Z_test = pca.transform(scaler.transform(emb_test)).astype(np.float32)
    result = scores_test.copy()
    for ci, clf in probe_models.items():
        prev, next_, mean, max_, std = build_seq_features(scores_test[:, ci])
        X_test = np.hstack([Z_test, scores_test[:, ci:ci+1],
                            prev[:, None], next_[:, None],
                            mean[:, None], max_[:, None], std[:, None]])
        prob  = clf.predict_proba(X_test)[:, 1].astype(np.float32)
        logit = np.log(prob + 1e-7) - np.log(1 - prob + 1e-7)
        result[:, ci] = (1 - alpha_blend) * scores_test[:, ci] + alpha_blend * logit
    return result

print('MLP probes with clip augmentation ready')

MLP probes with clip augmentation ready


In [10]:
# LightProtoSSM — same as exp13

class SelectiveSSM(nn.Module):
    def __init__(self, d_model, d_state=16, d_conv=4):
        super().__init__()
        self.d_model = d_model; self.d_state = d_state
        self.in_proj  = nn.Linear(d_model, 2 * d_model, bias=False)
        self.conv1d   = nn.Conv1d(d_model, d_model, d_conv, padding=d_conv-1, groups=d_model)
        self.dt_proj  = nn.Linear(d_model, d_model, bias=True)
        A = torch.arange(1, d_state+1, dtype=torch.float32).unsqueeze(0).expand(d_model, -1)
        self.A_log = nn.Parameter(torch.log(A)); self.D = nn.Parameter(torch.ones(d_model))
        self.B_proj = nn.Linear(d_model, d_state, bias=False)
        self.C_proj = nn.Linear(d_model, d_state, bias=False)

    def forward(self, x):
        B_sz, T, D = x.shape
        xz = self.in_proj(x); x_ssm, z = xz.chunk(2, dim=-1)
        x_c = self.conv1d(x_ssm.transpose(1,2))[:,:,:T].transpose(1,2)
        x_c = F.silu(x_c)
        dt  = F.softplus(self.dt_proj(x_c))
        A   = -torch.exp(self.A_log)
        B_  = self.B_proj(x_c); C = self.C_proj(x_c)
        h   = torch.zeros(B_sz, D, self.d_state, device=x.device)
        ys  = []
        for t in range(T):
            h = h * torch.exp(A[None] * dt[:,t,:,None]) + x[:,t,:,None] * (dt[:,t,:,None] * B_[:,t,None,:])
            ys.append((h * C[:,t,None,:]).sum(-1))
        return torch.stack(ys, dim=1) + x * self.D[None, None, :]


class LightProtoSSM(nn.Module):
    def __init__(self, d_input=1536, d_model=128, d_state=16, n_ssm_layers=2,
                 n_classes=234, n_windows=N_WINDOWS, dropout=0.15,
                 n_sites=20, meta_dim=16, cross_attn_heads=2):
        super().__init__()
        self.n_classes = n_classes; self.n_windows = n_windows
        self.input_proj = nn.Sequential(
            nn.Linear(d_input, d_model), nn.LayerNorm(d_model), nn.GELU(), nn.Dropout(dropout))
        self.pos_enc   = nn.Parameter(torch.randn(1, n_windows, d_model) * 0.02)
        self.site_emb  = nn.Embedding(n_sites, meta_dim)
        self.hour_emb  = nn.Embedding(24, meta_dim)
        self.meta_proj = nn.Linear(2 * meta_dim, d_model)
        self.ssm_fwd   = nn.ModuleList([SelectiveSSM(d_model, d_state) for _ in range(n_ssm_layers)])
        self.ssm_bwd   = nn.ModuleList([SelectiveSSM(d_model, d_state) for _ in range(n_ssm_layers)])
        self.ssm_merge = nn.ModuleList([nn.Linear(2 * d_model, d_model) for _ in range(n_ssm_layers)])
        self.ssm_norm  = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(n_ssm_layers)])
        self.drop      = nn.Dropout(dropout)
        self.cross_attn = nn.ModuleList([
            nn.MultiheadAttention(d_model, num_heads=cross_attn_heads, dropout=dropout, batch_first=True)
            for _ in range(n_ssm_layers)])
        self.cross_norm = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(n_ssm_layers)])
        self.prototypes  = nn.Parameter(torch.randn(n_classes, d_model) * 0.02)
        self.proto_temp  = nn.Parameter(torch.tensor(5.0))
        self.class_bias  = nn.Parameter(torch.zeros(n_classes))
        self.fusion_alpha = nn.Parameter(torch.zeros(n_classes))

    def init_prototypes(self, emb_tensor, labels_tensor):
        with torch.no_grad():
            h = self.input_proj(emb_tensor)
            for c in range(self.n_classes):
                mask = labels_tensor[:, c] > 0.5
                if mask.sum() > 0:
                    self.prototypes.data[c] = F.normalize(h[mask].mean(0), dim=0)

    def forward(self, emb, perch_logits=None, site_ids=None, hours=None):
        B, T, _ = emb.shape
        h = self.input_proj(emb) + self.pos_enc[:, :T, :]
        if site_ids is not None and hours is not None:
            meta = self.meta_proj(torch.cat([
                self.site_emb(site_ids.clamp(0, self.site_emb.num_embeddings-1)),
                self.hour_emb(hours.clamp(0, 23))], dim=-1))
            h = h + meta[:, None, :]
        for fwd, bwd, merge, norm, ca, cn in zip(
                self.ssm_fwd, self.ssm_bwd, self.ssm_merge, self.ssm_norm,
                self.cross_attn, self.cross_norm):
            res = h
            h_f = fwd(h); h_b = bwd(h.flip(1)).flip(1)
            h   = self.drop(merge(torch.cat([h_f, h_b], dim=-1)))
            h   = norm(h + res)
            attn_out, _ = ca(h, h, h); h = cn(h + attn_out)
        h_n = F.normalize(h, dim=-1); p_n = F.normalize(self.prototypes, dim=-1)
        sim = torch.matmul(h_n, p_n.T) * F.softplus(self.proto_temp) + self.class_bias[None,None,:]
        if perch_logits is not None:
            alpha = torch.sigmoid(self.fusion_alpha)[None, None, :]
            return alpha * sim + (1.0 - alpha) * perch_logits
        return sim


def train_proto_ssm(emb_full, scores_full, Y_full, meta_full, n_epochs=30, patience=8, lr=5e-4, n_sites=20):
    n_files = len(emb_full) // N_WINDOWS
    emb_f = emb_full.reshape(n_files, N_WINDOWS, -1)
    log_f = scores_full.reshape(n_files, N_WINDOWS, -1)
    lab_f = Y_full.reshape(n_files, N_WINDOWS, -1).astype(np.float32)
    fnames  = meta_full.groupby('filename', sort=False).size().index.tolist()
    sites_u = sorted(meta_full['site'].astype(str).unique())
    site2i  = {s: i+1 for i, s in enumerate(sites_u)}
    site_ids = np.array([min(site2i.get(str(meta_full.loc[meta_full['filename']==fn,'site'].iloc[0]), 0), n_sites-1) for fn in fnames], dtype=np.int64)
    hour_ids = np.array([int(meta_full.loc[meta_full['filename']==fn,'hour_utc'].iloc[0]) % 24 for fn in fnames], dtype=np.int64)
    model = LightProtoSSM(d_model=CFG['d_model'], d_state=CFG['d_state'],
                          n_ssm_layers=CFG['n_ssm_layers'], n_classes=N_CLASSES,
                          n_windows=N_WINDOWS, dropout=CFG['dropout'],
                          n_sites=n_sites, meta_dim=CFG['meta_dim'])
    model.init_prototypes(torch.tensor(emb_full, dtype=torch.float32), torch.tensor(Y_full, dtype=torch.float32))
    emb_t = torch.tensor(emb_f, dtype=torch.float32)
    log_t = torch.tensor(log_f, dtype=torch.float32)
    lab_t = torch.tensor(lab_f, dtype=torch.float32)
    site_t = torch.tensor(site_ids, dtype=torch.long)
    hour_t = torch.tensor(hour_ids, dtype=torch.long)
    pos_cnt = lab_t.sum(dim=(0,1))
    pos_weight = ((lab_t.shape[0]*lab_t.shape[1] - pos_cnt) / (pos_cnt+1)).clamp(max=CFG['pos_weight_cap'])
    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=CFG['weight_decay'])
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, epochs=n_epochs, steps_per_epoch=1, pct_start=0.3, anneal_strategy='cos')
    best_loss = float('inf'); best_state = None; wait = 0
    for ep in range(n_epochs):
        model.train()
        out  = model(emb_t, log_t, site_ids=site_t, hours=hour_t)
        loss = (focal_bce(out, lab_t, gamma=CFG['focal_gamma'], pos_weight=pos_weight[None,None,:])
                + CFG['distill_weight'] * F.mse_loss(out, log_t))
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sched.step()
        if loss.item() < best_loss:
            best_loss = loss.item()
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}; wait = 0
        else:
            wait += 1
        if wait >= patience: break
    if best_state: model.load_state_dict(best_state)
    model.eval(); return model, site2i

print('LightProtoSSM ready')

LightProtoSSM ready


In [11]:
# Pre-compute PCA for clip embeddings (global, not fold-dependent)
# We fit PCA on ALL soundscape embeddings, then transform clip embeddings
global_scaler = StandardScaler().fit(emb_tr)
global_pca    = PCA(n_components=CFG['pca_dim']).fit(global_scaler.transform(emb_tr))
clip_emb_pca  = global_pca.transform(global_scaler.transform(clip_emb)).astype(np.float32)
print(f'Clip PCA: {clip_emb.shape} → {clip_emb_pca.shape}')
print(f'  Explained variance: {global_pca.explained_variance_ratio_.sum():.2%}')

Clip PCA: (1032, 1536) → (1032, 64)
  Explained variance: 81.47%


In [12]:
# Full OOF pipeline
prior_tables = build_prior_tables(sc, Y_SC)
file_meta    = meta_tr.drop_duplicates('filename').reset_index(drop=True)
gkf          = GroupKFold(n_splits=N_OOF_SPLITS)

oof_proto = np.zeros((len(sc_tr), N_CLASSES), dtype=np.float32)
oof_mlp   = np.zeros((len(sc_tr), N_CLASSES), dtype=np.float32)
oof_prior = np.zeros((len(sc_tr), N_CLASSES), dtype=np.float32)
fold_aucs = []; fold_cmaps = []

global_class_weights = build_class_freq_weights(Y_FULL_aligned, cap=10.0)

for fold, (tr_f, va_f) in enumerate(
        gkf.split(file_meta, groups=file_meta['filename']), 1):
    t_fold = time.time()
    tr_fnames = set(file_meta.iloc[tr_f]['filename'])
    va_fnames = set(file_meta.iloc[va_f]['filename'])
    tr_mask   = meta_tr['filename'].isin(tr_fnames).values
    va_mask   = meta_tr['filename'].isin(va_fnames).values

    print(f'\nFold {fold}/{N_OOF_SPLITS} — train: {len(tr_fnames)}, val: {len(va_fnames)}')

    # ProtoSSM
    proto_model, site2i = train_proto_ssm(
        emb_tr[tr_mask], sc_tr[tr_mask], Y_FULL_aligned[tr_mask],
        meta_tr[tr_mask].reset_index(drop=True))

    n_va = int(va_mask.sum()) // N_WINDOWS
    va_fn_list  = meta_tr[va_mask].drop_duplicates('filename')['filename'].tolist()
    va_site_ids = np.array([min(site2i.get(meta_tr.loc[meta_tr['filename']==fn,'site'].iloc[0], 0), 19) for fn in va_fn_list], dtype=np.int64)
    va_hour_ids = np.array([int(meta_tr.loc[meta_tr['filename']==fn,'hour_utc'].iloc[0]) % 24 for fn in va_fn_list], dtype=np.int64)

    proto_model.eval()
    with torch.no_grad():
        proto_va = proto_model(
            torch.tensor(emb_tr[va_mask].reshape(n_va, N_WINDOWS, -1), dtype=torch.float32),
            torch.tensor(sc_tr[va_mask].reshape(n_va, N_WINDOWS, -1), dtype=torch.float32),
            site_ids=torch.tensor(va_site_ids, dtype=torch.long),
            hours=torch.tensor(va_hour_ids, dtype=torch.long),
        ).numpy().reshape(-1, N_CLASSES)

    # MLP probes with clip augmentation
    tr_emb = emb_tr[tr_mask]; tr_sc = sc_tr[tr_mask]; tr_Y = Y_FULL_aligned[tr_mask]
    scaler_fold = StandardScaler().fit(tr_emb)
    pca_fold    = PCA(n_components=CFG['pca_dim']).fit(scaler_fold.transform(tr_emb))
    Z_fold      = pca_fold.transform(scaler_fold.transform(tr_emb)).astype(np.float32)
    print(f'  PCA: {tr_emb.shape} → {Z_fold.shape}  ({pca_fold.explained_variance_ratio_.sum():.2%})')

    # Project clip embeddings into fold's PCA space
    clip_emb_pca_fold = pca_fold.transform(scaler_fold.transform(clip_emb)).astype(np.float32)

    print(f'  Training MLP probes (with clip augmentation)...')
    probe_models = train_mlp_probes_with_clips(
        tr_emb, tr_sc, tr_Y,
        clip_emb_pca_fold, clip_sc, clip_labels,
        scaler_fold, pca_fold, global_class_weights,
        pca_dim=CFG['pca_dim'], alpha_blend=CFG['alpha_blend'],
        min_pos=CFG['min_pos'], clip_repeat=CFG['clip_repeat_factor'])
    print(f'  Trained {len(probe_models)} probes')

    sc_va_mlp = apply_mlp_probes(emb_tr[va_mask], sc_tr[va_mask], probe_models,
                                  scaler_fold, pca_fold, CFG['alpha_blend'])

    # Fold-safe prior
    sc_clean_tr = sc[sc['filename'].isin(tr_fnames)].reset_index(drop=True)
    Y_clean_tr  = Y_SC[sc[sc['filename'].isin(tr_fnames)].index.tolist()]
    prior_fold  = build_prior_tables(sc_clean_tr, Y_clean_tr)
    sc_va_prior = apply_prior(
        sc_tr[va_mask],
        sites=meta_tr[va_mask]['site'].to_numpy(),
        hours=meta_tr[va_mask]['hour_utc'].to_numpy(),
        tables=prior_fold, lambda_prior=CFG['lambda_prior'])

    oof_proto[va_mask] = proto_va
    oof_mlp[va_mask]   = sc_va_mlp
    oof_prior[va_mask] = sc_va_prior

    first_pass = 0.20 * proto_va + 0.55 * sc_va_mlp + 0.25 * sc_va_prior
    probs_va   = sigmoid(first_pass / temperatures[None, :])
    probs_va   = file_confidence_scale(probs_va, power=0.05)
    probs_va   = adaptive_delta_smooth(probs_va, base_alpha=0.10)
    probs_va   = np.clip(probs_va, 0.0, 1.0)

    fold_auc  = macro_auc(Y_FULL_aligned[va_mask], probs_va)
    fold_cmap = padded_cmap(Y_FULL_aligned[va_mask], probs_va)
    fold_aucs.append(fold_auc); fold_cmaps.append(fold_cmap)
    print(f'  Fold {fold} AUC={fold_auc:.5f}  cmAP={fold_cmap:.5f}  ({time.time()-t_fold:.0f}s)')

print(f'\nMean AUC={np.mean(fold_aucs):.5f}  Mean cmAP={np.mean(fold_cmaps):.5f}')


Fold 1/3 — train: 39, val: 20


  PCA: (468, 1536) → (468, 64)  (84.62%)
  Training MLP probes (with clip augmentation)...


  Trained 45 probes
  Fold 1 AUC=0.89184  cmAP=0.95058  (21s)

Fold 2/3 — train: 39, val: 20


  PCA: (468, 1536) → (468, 64)  (83.71%)
  Training MLP probes (with clip augmentation)...


  Trained 53 probes
  Fold 2 AUC=0.89784  cmAP=0.97099  (21s)

Fold 3/3 — train: 40, val: 19


  PCA: (480, 1536) → (480, 64)  (84.51%)
  Training MLP probes (with clip augmentation)...


  Trained 51 probes
  Fold 3 AUC=0.87585  cmAP=0.96441  (21s)

Mean AUC=0.88851  Mean cmAP=0.96200


In [13]:
# Grid search + final OOF scores
best_cmap, best_wp, best_wm = 0.0, 0.20, 0.55
step = 0.05
results = []
for wp in np.arange(0.05, 0.50 + step, step):
    for wm in np.arange(0.30, 0.80 + step, step):
        wpr = 1.0 - wp - wm
        if wpr < 0.05 or wpr > 0.50: continue
        logits = wp * oof_proto + wm * oof_mlp + wpr * oof_prior
        probs  = sigmoid(logits / temperatures[None, :])
        probs  = adaptive_delta_smooth(probs, base_alpha=0.10)
        probs  = np.clip(probs, 0.0, 1.0)
        s = padded_cmap(Y_FULL_aligned, probs)
        results.append((s, wp, wm, wpr))
        if s > best_cmap:
            best_cmap = s; best_wp = wp; best_wm = wm

results.sort(reverse=True)
print('Top-5 blends:')
for s, wp, wm, wpr in results[:5]:
    print(f'  proto={wp:.2f}  mlp={wm:.2f}  prior={wpr:.2f}  cmAP={s:.5f}')
best_wpr = 1.0 - best_wp - best_wm

# Final OOF
logits_best = best_wp * oof_proto + best_wm * oof_mlp + best_wpr * oof_prior
oof_probs   = sigmoid(logits_best / temperatures[None, :])
oof_probs   = file_confidence_scale(oof_probs, power=0.05)
oof_probs   = adaptive_delta_smooth(oof_probs, base_alpha=0.10)
oof_probs   = np.clip(oof_probs, 0.0, 1.0)
oof_auc     = macro_auc(Y_FULL_aligned, oof_probs)
oof_cmap    = padded_cmap(Y_FULL_aligned, oof_probs)

# Fixed weights
logits_fixed = 0.20 * oof_proto + 0.55 * oof_mlp + 0.25 * oof_prior
probs_fixed  = sigmoid(logits_fixed / temperatures[None, :])
probs_fixed  = file_confidence_scale(probs_fixed, power=0.05)
probs_fixed  = adaptive_delta_smooth(probs_fixed, base_alpha=0.10)
probs_fixed  = np.clip(probs_fixed, 0.0, 1.0)
fixed_cmap   = padded_cmap(Y_FULL_aligned, probs_fixed)
fixed_auc    = macro_auc(Y_FULL_aligned, probs_fixed)

print(f'\nFixed weights (20/55/25): AUC={fixed_auc:.5f}  cmAP={fixed_cmap:.5f}')
print(f'Best blend:               AUC={oof_auc:.5f}  cmAP={oof_cmap:.5f}')
print(f'vs exp11 (0.9166): delta cmAP = {fixed_cmap - 0.9166:+.5f}')
print(f'Total wall time: {(time.time() - _WALL_START)/60:.1f} min')

Top-5 blends:
  proto=0.25  mlp=0.50  prior=0.25  cmAP=0.90874
  proto=0.30  mlp=0.50  prior=0.20  cmAP=0.90857
  proto=0.20  mlp=0.50  prior=0.30  cmAP=0.90852
  proto=0.35  mlp=0.50  prior=0.15  cmAP=0.90848
  proto=0.30  mlp=0.45  prior=0.25  cmAP=0.90845

Fixed weights (20/55/25): AUC=0.89513  cmAP=0.90891
Best blend:               AUC=0.89475  cmAP=0.90910
vs exp11 (0.9166): delta cmAP = -0.00769
Total wall time: 1.8 min
